## Customer Support Ticket Intelligence: Feature Engineering & Model Selection

Continues from the EDA notebook. Predicts two targets from ticket text: `queue` (department) and `priority` (urgency). Covers preprocessing, feature engineering, model comparison, and tuning for both - `_q` suffix for queue, `_p` for priority.

**From EDA:**
- `type` is associated with both targets, so it's used as a feature for both.
- `queue` is badly imbalanced (smallest class <100 of ~21k tickets); `priority` is only mildly imbalanced.

In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [2]:
df_new = pd.read_pickle("nlp_eda.pkl")
df_new.shape

(20982, 11)

In [3]:
df_new.sample(3)

,subject,body,answer,type,queue,priority,tag_1,tag_2,tag_3,tag_4,body_length
18577,Details on Medical Data Backup Services,Could you provide detailed information on your...,We offer customized backup services for medica...,Request,Returns and Exchanges,medium,Backup,Compliance,Medical,IT,154
12747,Issues with Invoices,"Overcharged due to a system error, requiring s...",Currently investigating the invoice problem re...,Problem,Billing and Payments,low,Billing,Payment,Bug,Tech Support,70
9064,Issues with Adobe Photoshop Integration,Issues with integrating Adobe Photoshop CC 202...,"Dear <name>, we apologize for the issues you a...",Problem,Technical Support,high,Bug,Disruption,Performance,Feature,177


### Text preprocessing

Clean HTML, URLs, emails, punctuation, casing; tokenize, drop stopwords, lemmatize. Same function for `subject` and `body`.

In [4]:
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

def nlpfunc(text):
    text = text.lower()

    # newline artifacts, HTML, URLs, emails
    text = re.sub(r"\\n+", " ", text)
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # punctuation, numbers, extra whitespace
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))

    negation_words = {
    "no", "nor", "not", "ain", "aren't", "couldn't", "didn't", "doesn't",
    "don't", "hadn't", "hasn't", "haven't", "out", "isn't", "mightn't", "mustn't",
    "needn't", "shan't", "shouldn't", "wasn't", "weren't", "won't", "wouldn't"
    }
    
    stop_words = stop_words - negation_words
    tokens = [token for token in tokens if token not in stop_words]

    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return ' '.join(tokens)

df_new["body_clean"] = df_new["body"].apply(nlpfunc)
df_new["subject_clean"] = df_new["subject"].apply(nlpfunc)

Cleaned text kept in new columns — raw text is still needed later for semantic search, where showing a real ticket back to the user matters.

In [5]:
df_new[["body", "body_clean"]].sample(5)

,body,body_clean
13013,Seeking information on security solutions for ...,seeking information security solution medical ...
2755,"Dear Customer Service, I would like to inquire...",dear customer service would like inquire integ...
12948,"Hello Customer Support, we are experiencing a ...",hello customer support experiencing system out...
9431,Customers encounter sync delays even after res...,customer encounter sync delay even restarting ...
14853,"Customer Support, seeking advice on integratin...",customer support seeking advice integrating th...


#### Side check: should "dear", "customer", "support" be removed too?

They show up in almost every ticket opener. Checked whether they vary by `queue` before removing — if they do, that's signal, not noise.

In [6]:
df_new["check_for_dear"] = df_new["body_clean"].str.contains("dear")
df_new.groupby("queue")["check_for_dear"].mean()

queue
Billing and Payments               0.177570
Customer Service                   0.170296
General Inquiry                    0.219512
Human Resources                    0.129496
IT Support                         0.162910
Product Support                    0.165026
Returns and Exchanges              0.151970
Sales and Pre-Sales                0.183386
Service Outages and Maintenance    0.191716
Technical Support                  0.179936
Name: check_for_dear, dtype: float64

In [7]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df_new["queue"], df_new["check_for_dear"])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(chi2, p_value)

21.74991890795992 0.009706257345017283


chi-square = 21.75, p = 0.0097 - technically significant, but per-queue rates stay flat (13%–22%). At ~21,000 tickets, even small effects reach significance without being useful.

**Decision:** leave them in, let TF-IDF's own weighting handle them.

### Combining subject and body

`subject` is a short summary, `body` has the detail - combined so one TF-IDF vectorizer sees both.

In [8]:
df_new["combine_text"] = df_new["subject_clean"] + " " + df_new["body_clean"]

##### Adding `type` as a feature - and why `priority` isn't used for `queue`

`type` (Incident/Problem/Request/Change) is associated with `queue` (chi-square) and known at intake, so it's a fair feature. `priority` is excluded here since it's itself a prediction target - using it would leak one model's errors into another. Same logic in reverse for the `priority` model.

In [9]:
df_new["type"].isna().sum()

np.int64(0)

### Feature set and target - `queue`

`X_q`: combined text + `type`. `y_q`: encoded `queue` label.

In [10]:
from sklearn.preprocessing import LabelEncoder

X_q = df_new[["combine_text", "type"]]

queue_encode = LabelEncoder()
y_q = queue_encode.fit_transform(df_new["queue"])

### Train/test split - `queue`

30% held out, stratified on `y_q`.

In [11]:
from sklearn.model_selection import train_test_split

x_train_q, x_test_q, y_train_q, y_test_q = train_test_split(
    X_q, y_q, test_size=0.30, stratify=y_q, random_state=42
)

x_train_q.shape, x_test_q.shape

((14687, 2), (6295, 2))

### Leakage-safe preprocessing

`ColumnTransformer` handles TF-IDF (`combine_text`) and one-hot (`type`) together, wrapped in a `Pipeline` so both refit fresh on each CV fold. Built as a function - every model needs its own unfitted copy.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

def build_preprocessor_queue():
    return ColumnTransformer(transformers=[
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2), "combine_text"),
        ("type_ohe", OneHotEncoder(handle_unknown="ignore"), ["type"]),
    ])

# sanity check: fit on train only, confirm shape
check_q = build_preprocessor_queue().fit_transform(x_train_q)
check_q.shape

(14687, 10004)

### Baseline model comparison - `queue`

5-fold stratified CV, compared on **macro-F1** (not accuracy, so small queues like Human Resources still count). `memory=memory` caches the preprocessing step across models.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold

import joblib
import tempfile

cache_dir = tempfile.mkdtemp()
memory = joblib.Memory(location=cache_dir, verbose=0)

models_q = {
    "log_reg": LogisticRegression(class_weight="balanced", max_iter=1000),
    "svm": LinearSVC(class_weight="balanced", random_state=42),
    "xgb": XGBClassifier(random_state=42),
    "naive": MultinomialNB(),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "forest": RandomForestClassifier(class_weight="balanced", random_state=42),
    "dummy": DummyClassifier(strategy="most_frequent", random_state=42),
}

cv_q = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

results_q = {}

for name, model in models_q.items():
    pipe = Pipeline([
        ("preprocess", build_preprocessor_queue()),
        ("clf", model),
    ], memory=memory)
    scores = cross_validate(pipe, x_train_q, y_train_q, cv=cv_q, scoring=scoring, n_jobs=-1)
    results_q[name] = {metric: scores[f"test_{metric}"].mean() for metric in scoring}

results_df_q = pd.DataFrame(results_q).T.sort_values("f1_macro", ascending=False)
results_df_q

,accuracy,precision_macro,recall_macro,f1_macro
forest,0.620888,0.709070,0.599178,0.634311
svm,0.538911,0.525176,0.561261,0.540144
xgb,0.522775,0.660814,0.415038,0.471107
log_reg,0.463063,0.426735,0.519441,0.455363
decision_tree,0.457139,0.430870,0.430422,0.429934
naive,0.399537,0.408918,0.214206,0.207369
dummy,0.296929,0.029693,0.100000,0.045790


**Random Forest wins clearly**, SVM a distant second. Everything else trails enough that only these two get tuned.

### Hyperparameter tuning - `queue`

In [14]:
# Random Forest tuning

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_params_q = {
    "clf__n_estimators": [100, 150, 200, 250],
    "clf__max_depth": [None, 40, 60],
    "clf__min_samples_split": [2, 3, 4, 8],
    "clf__min_samples_leaf": [1, 2, 3, 4, 5],
}

rf_pipe_q = Pipeline([
    ("preprocess", build_preprocessor_queue()),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
], memory=memory)

rf_search_q = RandomizedSearchCV(
    rf_pipe_q,
    param_distributions=rf_params_q,
    n_iter=50,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
)

rf_search_q.fit(x_train_q, y_train_q)
print(rf_search_q.best_score_, rf_search_q.best_params_)

0.6399256732918683 {'clf__n_estimators': 200, 'clf__min_samples_split': 3, 'clf__min_samples_leaf': 1, 'clf__max_depth': None}


In [15]:
# SVC tuning

from sklearn.svm import LinearSVC

svm_params_q = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__loss": ["hinge", "squared_hinge"],
}

svm_pipe_q = Pipeline([
    ("preprocess", build_preprocessor_queue()),
    ("clf", LinearSVC(class_weight="balanced", random_state=42)),
], memory=memory)

svm_search_q = RandomizedSearchCV(
    svm_pipe_q,
    param_distributions=svm_params_q,
    n_iter=50,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
)

svm_search_q.fit(x_train_q, y_train_q)
print(svm_search_q.best_score_, svm_search_q.best_params_)

0.552427054221621 {'clf__loss': 'squared_hinge', 'clf__C': 10}


### Final holdout check - `queue`

`x_test_q` untouched by tuning - the one honest final number.

In [16]:
from sklearn.metrics import classification_report, f1_score

final_pipe_q = Pipeline([
    ("preprocess", build_preprocessor_queue()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        min_samples_split=3,
        min_samples_leaf=1,
        max_depth=None,
        class_weight="balanced",
        random_state=42,
    )),
])

final_pipe_q.fit(x_train_q, y_train_q)
y_pred_q = final_pipe_q.predict(x_test_q)

print("Final test macro-F1:", f1_score(y_test_q, y_pred_q, average="macro"))
print(classification_report(y_test_q, y_pred_q, target_names=queue_encode.classes_))

Final test macro-F1: 0.6872127829823899
                                 precision    recall  f1-score   support

           Billing and Payments       0.90      0.82      0.86       610
               Customer Service       0.51      0.66      0.58       923
                General Inquiry       0.95      0.63      0.76        86
                Human Resources       0.92      0.58      0.71       125
                     IT Support       0.67      0.59      0.63       759
                Product Support       0.65      0.56      0.60      1158
          Returns and Exchanges       0.88      0.57      0.69       320
            Sales and Pre-Sales       0.76      0.56      0.65       191
Service Outages and Maintenance       0.61      0.87      0.72       254
              Technical Support       0.66      0.71      0.68      1869

                       accuracy                           0.66      6295
                      macro avg       0.75      0.66      0.69      6295
         

### Final Model: Random Forest for `queue`

Winner after comparing 7 models and tuning the top 2.

**Config:** TF-IDF (`ngram_range=(1,2)`, `max_features=10000`, `min_df=2`) + one-hot `type`, `RandomForestClassifier(n_estimators=200, min_samples_split=3, min_samples_leaf=1, max_depth=None, class_weight="balanced")`.

**Holdout: macro-F1 = 0.687**, accuracy = 0.66. Dummy baseline: 0.046 - beats it ~15x.

**Per-class:**
- `Human Resources` (smallest, ~86 tickets): 0.71 F1 - strong given the size.
- `Customer Service`: weakest at 0.58 F1 - likely overlaps with `General Inquiry`/`Product Support`.
- `Service Outages and Maintenance`: high recall (0.87), lower precision (0.61) - over-predicted.

Reflects real routing ambiguity and data limits on small classes, not a modeling shortfall.

### Predicting `priority`

Same pipeline as `queue`, feature choice flipped: `type` stays in (chi-square = 143.23, p ≈ 2×10⁻²⁸ - stronger than for `queue`), `queue` is excluded for the same leakage reason, reversed.

In [17]:
df_new["priority"].value_counts()

priority
medium    8494
high      8214
low       4274
Name: count, dtype: int64

Only 3 levels, much milder imbalance than `queue`: `low` (4,274) is about half of `medium`/`high` (~8,300–8,500). `class_weight="balanced"` kept, but the effect is smaller.

In [18]:
X_p = df_new[["combine_text", "type"]]

priority_encode = LabelEncoder()
y_p = priority_encode.fit_transform(df_new["priority"])

In [19]:
x_train_p, x_test_p, y_train_p, y_test_p = train_test_split(
    X_p, y_p, test_size=0.30, stratify=y_p, random_state=42
)

x_train_p.shape, x_test_p.shape

((14687, 2), (6295, 2))

Same `ColumnTransformer` structure as `queue`, own function so every model gets a fresh unfitted copy.

In [20]:
def build_preprocessor_priority():
    return ColumnTransformer(transformers=[
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2), "combine_text"),
        ("type_ohe", OneHotEncoder(handle_unknown="ignore"), ["type"]),
    ])

check_p = build_preprocessor_priority().fit_transform(x_train_p)
check_p.shape

(14687, 10004)

### Baseline model comparison - `priority`

Same models, CV, and caching as `queue`.

In [21]:
cache_dir_p = tempfile.mkdtemp()
memory_p = joblib.Memory(location=cache_dir_p, verbose=0)

models_p = {
    "log_reg": LogisticRegression(class_weight="balanced", max_iter=1000),
    "svm": LinearSVC(class_weight="balanced", random_state=42),
    "xgb": XGBClassifier(random_state=42),
    "naive": MultinomialNB(),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "forest": RandomForestClassifier(class_weight="balanced", random_state=42),
    "dummy": DummyClassifier(strategy="most_frequent", random_state=42),
}

cv_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_p = {}

for name, model in models_p.items():
    pipe = Pipeline([
        ("preprocess", build_preprocessor_priority()),
        ("clf", model),
    ], memory=memory_p)
    scores = cross_validate(pipe, x_train_p, y_train_p, cv=cv_p, scoring=scoring, n_jobs=-1)
    results_p[name] = {metric: scores[f"test_{metric}"].mean() for metric in scoring}

results_df_p = pd.DataFrame(results_p).T.sort_values("f1_macro", ascending=False)
results_df_p

,accuracy,precision_macro,recall_macro,f1_macro
forest,0.671274,0.677881,0.646277,0.656952
svm,0.590319,0.574783,0.576782,0.575617
log_reg,0.554097,0.540833,0.552085,0.543229
xgb,0.566555,0.592619,0.509614,0.512265
decision_tree,0.521415,0.504262,0.506790,0.505193
naive,0.501258,0.542227,0.427575,0.398814
dummy,0.404780,0.134927,0.333333,0.192096


**Random Forest wins again**, same ranking as `queue`. Same call: tune RF and SVM, skip the rest.

In [22]:
rf_params_p = {
    "clf__n_estimators": [100, 150, 200, 250],
    "clf__max_depth": [None, 40, 60],
    "clf__min_samples_split": [2, 3, 4, 8],
    "clf__min_samples_leaf": [1, 2, 3, 4, 5],
}

rf_pipe_p = Pipeline([
    ("preprocess", build_preprocessor_priority()),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
], memory=memory_p)

rf_search_p = RandomizedSearchCV(
    rf_pipe_p,
    param_distributions=rf_params_p,
    n_iter=50,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
)

rf_search_p.fit(x_train_p, y_train_p)
print(rf_search_p.best_score_, rf_search_p.best_params_)

0.6653304907128013 {'clf__n_estimators': 200, 'clf__min_samples_split': 4, 'clf__min_samples_leaf': 1, 'clf__max_depth': None}


In [23]:
svm_params_p = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__loss": ["hinge", "squared_hinge"],
}

svm_pipe_p = Pipeline([
    ("preprocess", build_preprocessor_priority()),
    ("clf", LinearSVC(class_weight="balanced", random_state=42)),
], memory=memory_p)

svm_search_p = RandomizedSearchCV(
    svm_pipe_p,
    param_distributions=svm_params_p,
    n_iter=50,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
)

svm_search_p.fit(x_train_p, y_train_p)
print(svm_search_p.best_score_, svm_search_p.best_params_)

0.5731396208680469 {'clf__loss': 'squared_hinge', 'clf__C': 1}


### Final holdout check - `priority`

In [24]:
final_pipe_p = Pipeline([
    ("preprocess", build_preprocessor_priority()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        min_samples_split=3,
        min_samples_leaf=1,
        max_depth=None,
        class_weight="balanced",
        random_state=42,
    )),
])

final_pipe_p.fit(x_train_p, y_train_p)
y_pred_p = final_pipe_p.predict(x_test_p)

print("Final test macro-F1:", f1_score(y_test_p, y_pred_p, average="macro"))
print(classification_report(y_test_p, y_pred_p, target_names=priority_encode.classes_))

Final test macro-F1: 0.7013963145239456
              precision    recall  f1-score   support

        high       0.73      0.75      0.74      2464
         low       0.75      0.57      0.65      1282
      medium       0.69      0.74      0.71      2549

    accuracy                           0.71      6295
   macro avg       0.72      0.69      0.70      6295
weighted avg       0.72      0.71      0.71      6295



### Final Model: Random Forest for `priority`

Same approach and winner as `queue`.

**Config:** same as `queue` - TF-IDF (`max_features=10000`) + one-hot `type`, `RandomForestClassifier(n_estimators=200, min_samples_split=3, min_samples_leaf=1, max_depth=None, class_weight="balanced")`.

**Holdout: macro-F1 = 0.701**, accuracy = 0.71.

**Per-class:**
- `low`: weakest (0.65 F1, recall 0.57) - least distinctive tone, easy to confuse with medium.
- `high`/`medium`: comparable (0.74 / 0.71 F1).

**`queue` vs `priority`:** priority's raw macro-F1 is higher, but mostly because it's an easier problem (3 balanced classes vs. 10 with a 20x imbalance). Against each one's own dummy baseline, `queue` beats its floor by ~15x vs. `priority`'s ~3.7x - `queue` is the more strongly learned signal, since topic keywords are more distinctive than urgency tone.

### Next: semantic search

Embed all tickets; for a new one, retrieve the closest past tickets and show how they were resolved.

`search_vectorizer` fits on all of `combine_text`, not just `x_train` - no train/test split needed here since retrieval isn't evaluated for generalization, just indexing every ticket that exists.

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

search_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
search_matrix = search_vectorizer.fit_transform(df_new["combine_text"])

search_matrix.shape

(20982, 10000)

Clean a new ticket the same way as training data, embed it, rank all tickets by cosine similarity. Returns the closest matches plus how they were resolved (`answer`) - showing what happened last time, not just a label.

In [26]:
def find_similar_tickets(new_text, top_n=5):
    cleaned = nlpfunc(new_text)
    query_vec = search_vectorizer.transform([cleaned])
    
    similarities = cosine_similarity(query_vec, search_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = df_new.iloc[top_indices][["subject", "body", "queue", "priority", "answer"]].copy()
    results["similarity"] = similarities[top_indices]
    
    return results

In [27]:
example = "I was running the software, and it freezed and crashed out"

similar = find_similar_tickets(example, top_n=5)
for idx, row in similar.iterrows():
    print(f"[sim={row['similarity']:.3f}] queue={row['queue']} priority={row['priority']}")
    print(f"  {row['body'][:200]}")
    print(f"  RESOLUTION: {row['answer'][:200]}")
    print("-" * 80)

[sim=0.387] queue=Product Support priority=medium
  The data upload software has crashed.
  RESOLUTION: Sorry for the issue with the Finanzanalytik software update. Could you please provide more details about the error or crash? This will help us assist you in resolving the data upload problem.
--------------------------------------------------------------------------------
[sim=0.277] queue=Technical Support priority=medium
  The system crashed while running Keras models for investment predictions. Restarting and reinstalling resolved the issue.
  RESOLUTION: To assist with your Keras model issue, could you please provide the details of the system error message you received? This will help guide the next steps to resolve the issue. To schedule a call at a 
--------------------------------------------------------------------------------
[sim=0.237] queue=Product Support priority=high
  The application crashed during data processing due to compatibility issues.
  RESOLUTION: We apologiz

Tested on "My software is not running" (no exact duplicate in the data) - top hits are genuinely about software crashing, and queue/priority look sensible. Works because the query shares real vocabulary with matching tickets - also its limitation, tested next.

#### Testing pretrained embeddings against TF-IDF

TF-IDF only matches literal shared words. Real test: a query with little vocabulary overlap with the ticket it should still find - where embeddings should have the edge.

Used pretrained GloVe vectors, averaged per ticket, same cosine similarity as TF-IDF. (gensim kept breaking on scipy compatibility, so spaCy's `en_core_web_md` - same underlying GloVe vectors - was used instead.)

In [28]:
import spacy

nlp = spacy.load("en_core_web_md")

def embed_text(text, model=None, dim=300):  # en_core_web_md vectors are 300-dim, not 100
    doc = nlp(text)
    if not doc.has_vector or doc.vector_norm == 0:
        return np.zeros(dim)
    return doc.vector

glove_matrix = np.vstack(df_new["combine_text"].apply(lambda t: embed_text(t)))
glove_matrix.shape

(20982, 300)

In [29]:
def find_similar_tickets_glove(new_text, top_n=5):
    cleaned = nlpfunc(new_text)
    query_vec = embed_text(cleaned).reshape(1, -1)

    similarities = cosine_similarity(query_vec, glove_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]

    results = df_new.iloc[top_indices][["subject", "body", "queue", "priority", "answer"]].copy()
    results["similarity"] = similarities[top_indices]
    return results

In [30]:
def compare_retrieval(query, top_n=3):
    print(f"QUERY: {query}\n")
    print("=== TF-IDF ===")
    for idx, row in find_similar_tickets(query, top_n).iterrows():
        print(f"[sim={row['similarity']:.3f}] {row['body'][:120]}")
    print("\n=== GloVe ===")
    for idx, row in find_similar_tickets_glove(query, top_n).iterrows():
        print(f"[sim={row['similarity']:.3f}] {row['body'][:120]}")
    print("=" * 90)

compare_retrieval("my app freezes constantly and won't respond")
compare_retrieval("I was billed twice for the same month")
compare_retrieval("nobody has answered my emails about this problem")

QUERY: my app freezes constantly and won't respond

=== TF-IDF ===
[sim=0.434] System freeze happened while running data analytics reports
[sim=0.291] There is a critical bug that is causing the project management app to crash on both iOS and Android devices after recent
[sim=0.286] An unexpected system freeze happened while performing project updates, which might be related to compatibility issues ca

=== GloVe ===
[sim=0.877] Dear Customer Support Team,

I hope this message finds you well. I am reaching out to voice my concerns regarding the Le
[sim=0.868] Dear Customer Support Team, I hope this message finds you in good spirits. I am writing to ask for a detailed invoice fo
[sim=0.866] Customers are facing issues with Evernote and Shopify apps. Please acknowledge the issue and provide details on the erro
QUERY: I was billed twice for the same month

=== TF-IDF ===
[sim=0.348] The project management subscription was billed twice, possibly due to a system error during renewal.
[sim=0.

#### Result: TF-IDF wins, and it's not close

On low-overlap queries, GloVe's top hits are generic boilerplate, with similarity scores clustered tightly (~0.86–0.87) — it barely distinguishes tickets. TF-IDF stays on-topic.

Same root cause as the "dear" chi-square finding: TF-IDF's IDF downweights boilerplate automatically; a plain average of word vectors doesn't, so "dear", "please", "could" dilute the signal as much as the words that matter.

**Sticking with TF-IDF for retrieval.** An IDF-weighted embedding average might close some of the gap, but that's out of scope here.

### Model Evaluation

In [31]:
sub = input("Enter the Subject.")

query = input("What is your query?")


print("Please select the type of your problem. " \
"1. Incident " \
"2. Request " \
"3. Problem " \
"4. Change ")

ticket = int(input())

Please select the type of your problem. 1. Incident 2. Request 3. Problem 4. Change 


In [33]:
if ticket == 1:
    ticket_type = "Incident"
elif ticket == 2:
    ticket_type = "Request"
elif ticket == 3:
    ticket_type = "Problem"
else:
    ticket_type = "Change"


def clean(sub, query):
    clean_subject = nlpfunc(sub)
    clean_query = nlpfunc(query)
    return clean_subject + " " + clean_query

clean_txt = clean(sub, query)

# Build a one-row DataFrame matching the training feature structure exactly
new_ticket = pd.DataFrame({
    "combine_text": [clean_txt],
    "type": [ticket_type],
})

print("Query:", query, "\n")

#making a prediction - queue and Priority
predicted_queue = queue_encode.inverse_transform(final_pipe_q.predict(new_ticket))
predicted_priority = priority_encode.inverse_transform(final_pipe_p.predict(new_ticket))

#printing the prediciton
print("Predicted queue:", predicted_queue[0])
print("Predicted priority:", predicted_priority[0], "\n")

#Finding similar tickets

search_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2)
search_matrix = search_vectorizer.fit_transform(df_new["combine_text"])

def find_similar_tickets(new_text, top_n=5):
    cleaned = nlpfunc(new_text)
    query_vec = search_vectorizer.transform([cleaned])
    
    similarities = cosine_similarity(query_vec, search_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = df_new.iloc[top_indices][["subject", "body", "queue", "priority", "answer"]].copy()
    results["similarity"] = similarities[top_indices]
    
    return results


similar = find_similar_tickets(clean_txt, top_n=5)
for idx, row in similar.iterrows():
    print(f"[sim={row['similarity']:.3f}] queue={row['queue']} priority={row['priority']}", "\n")
    print(f"Ticket:  {row['body'][:300]}")
    print(f"  RESOLUTION: {row['answer'][:300]}")
    print("-" * 80)

Query: Purchased the new software, but it's laggy. Please take a look. 

Predicted queue: Technical Support
Predicted priority: high 

[sim=0.367] queue=Customer Service priority=low 

Ticket:  Please integrate new software tools to enhance our project management.
  RESOLUTION: Your request to integrate new software tools to improve project management has been received. We need more specific information about the tools you are interested in. Would it be convenient to call and discuss the details of your request further?
--------------------------------------------------------------------------------
[sim=0.313] queue=Technical Support priority=medium 

Ticket:  Could you provide more information on the digital strategy services related to technical support, particularly focusing on product integration? This would help us better understand how these services can meet our technical support needs.
  RESOLUTION: Dear [Name], thank you for your interest in our digital strategy technical sup

### Summary

- `queue`: macro-F1 = 0.687 (Random Forest, ~15x the dummy baseline).
- `priority`: macro-F1 = 0.701 (same model/config, ~3.7x its dummy baseline) — `queue`'s bigger relative lift means it's the more strongly learned target, despite the lower raw score.
- Retrieval: TF-IDF beat a GloVe-embedding baseline clearly, since IDF weighting handles boilerplate that plain vector averaging can't.
- Demo function above ties it together: predicts `queue`/`priority` for a new ticket and retrieves similar past tickets with their resolutions.

### Saving artifacts for deployment

Both classifiers, both label encoders, the TF-IDF search vectorizer + matrix, and a slim lookup table (subject/body/queue/priority/answer) are saved with `joblib` so a separate Streamlit app can load them without needing this notebook.

In [ ]:
import joblib
import os

save_dir = "customer_support_app"  # adjust to wherever you want the app to live
os.makedirs(save_dir, exist_ok=True)

joblib.dump(final_pipe_q, os.path.join(save_dir, "final_pipe_q.pkl"))
joblib.dump(final_pipe_p, os.path.join(save_dir, "final_pipe_p.pkl"))
joblib.dump(queue_encode, os.path.join(save_dir, "queue_encode.pkl"))
joblib.dump(priority_encode, os.path.join(save_dir, "priority_encode.pkl"))
joblib.dump(search_vectorizer, os.path.join(save_dir, "search_vectorizer.pkl"))
joblib.dump(search_matrix, os.path.join(save_dir, "search_matrix.pkl"))

search_df = df_new[["subject", "body", "queue", "priority", "answer"]]
joblib.dump(df_lookup, os.path.join(save_dir, "search_df.pkl"))

print(os.listdir(save_dir))

['df_lookup.pkl', 'final_pipe_p.pkl', 'final_pipe_q.pkl', 'priority_encode.pkl', 'queue_encode.pkl', 'search_matrix.pkl', 'search_vectorizer.pkl']
